# Building Multi-Agent Systems

**Author:** Shuvam Banerji Seal

**Last Updated:** April 2026

## Learning Objectives

By the end of this notebook, you will:

1. Understand multi-agent architecture patterns
2. Design agent communication protocols
3. Implement coordination strategies
4. Handle agent failures and recovery
5. Scale multi-agent systems

## What Are Multi-Agent Systems?

Multi-agent systems consist of multiple autonomous agents that:

1. **Operate independently** with their own goals
2. **Interact** with each other and environment
3. **Cooperate** to achieve shared objectives
4. **Negotiate** when conflicts arise
5. **Emerge** with collective intelligence

## Architecture Patterns

### 1. **Hierarchical Architecture**

```
        ┌──────────────────┐
        │  Master Agent    │
        │  (Coordinator)   │
        └────────┬─────────┘
                 │
    ┌────────────┼────────────┐
    │            │            │
    ▼            ▼            ▼
┌─────────┐  ┌─────────┐  ┌─────────┐
│ Worker  │  │ Worker  │  │ Worker  │
│ Agent 1 │  │ Agent 2 │  │ Agent 3 │
└─────────┘  └─────────┘  └─────────┘
```

**Use case**: Task delegation, project management

### 2. **Peer-to-Peer Architecture**

```
    ┌─────────┐      ┌─────────┐
    │ Agent 1 │◄────►│ Agent 2 │
    └────┬────┘      └────┬────┘
         │                │
    ┌────▼────┐      ┌────▼────┐
    │ Agent 3 │◄────►│ Agent 4 │
    └─────────┘      └─────────┘
```

**Use case**: Distributed systems, consensus

### 3. **Hybrid Architecture**

```
        ┌──────────────────┐
        │  Coordinator     │
        │  (Central Hub)   │
        └────────┬─────────┘
                 │
    ┌────────────┼────────────┐
    │            │            │
    ▼            ▼            ▼
┌─────────┐  ┌─────────┐  ┌─────────┐
│ Agent 1 │◄─┼────────►│ Agent 2 │  
└────┬────┘  │         └────┬────┘
     │       │              │
     └───────┼──────────────┘
             ▼
        ┌─────────┐
        │ Agent 3 │
        └─────────┘
```

**Use case**: Complex workflows with both hierarchy and peer communication

## Agent Communication Protocols

### Message Types

1. **Request**: One agent asks another to perform action
2. **Response**: Reply to a request
3. **Broadcast**: Send to all agents
4. **Event**: Notification of state change
5. **Query**: Information request

### Communication Patterns

```python
# Synchronous: Sender waits for response
response = await agent.request_action(action_spec)

# Asynchronous: Sender continues
task_id = agent.request_action_async(action_spec)

# Publish-Subscribe: Event-driven
agent.subscribe("task_completed", callback)
```

## Coordination Strategies

### 1. **Sequential Orchestration**

Agents execute one after another:

```
Task → [Agent A] → [Agent B] → [Agent C] → Result
```

**Code Example:**
```python
result = await agent_a.execute(task)
result = await agent_b.execute(result)
result = await agent_c.execute(result)
```

### 2. **Parallel Execution**

Agents work simultaneously:

```
        → [Agent A] → 
       /              \
Task -→  [Agent B] → Result
       \              /
        → [Agent C] →
```

**Code Example:**
```python
results = await asyncio.gather(
    agent_a.execute(task),
    agent_b.execute(task),
    agent_c.execute(task)
)
final_result = aggregate(results)
```

### 3. **Conditional Branching**

Agent decides which agent to invoke next:

```
      → [Financial Agent] →
     /                      \
Task -→ [Router Agent] → → Result
     \                      /
      → [Legal Agent] →
```

## Real-World Example: Research Team

A multi-agent system for research:

```
User Query
    │
    ▼
┌──────────────────┐
│ Router Agent     │  → Analyzes query
│ (LangChain)      │
└────────┬─────────┘
         │
  ┌──────┴───────┐
  │              │
  ▼              ▼
┌────────────────────┐  ┌──────────────────┐
│ Search Agent       │  │ Analysis Agent   │
│ (AGNO)             │  │ (AGNO)           │
│ - Web search       │  │ - Sentiment      │
│ - Sources          │  │ - Themes         │
└────────┬───────────┘  └──────────┬───────┘
         │                         │
         └────────────┬────────────┘
                      │
                      ▼
              ┌───────────────┐
              │ Synthesis     │
              │ Agent         │
              │ (AGNO)        │
              └───────┬───────┘
                      │
                      ▼
              Final Report
```

## Error Handling and Recovery

### Resilience Strategies

#### 1. **Retry Logic**
```python
async def execute_with_retry(agent, task, max_retries=3):
    for attempt in range(max_retries):
        try:
            return await agent.execute(task)
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            await asyncio.sleep(2 ** attempt)  # Exponential backoff
```

#### 2. **Graceful Degradation**
```python
# If specialized agent fails, use general agent
try:
    result = await specialized_agent.execute(task)
except Exception:
    result = await general_agent.execute(task)
```

#### 3. **Circuit Breaker**
```python
class CircuitBreaker:
    def __init__(self, threshold=5):
        self.failures = 0
        self.threshold = threshold
        self.open = False
    
    async def call(self, agent, task):
        if self.open:
            raise Exception("Circuit breaker is open")
        try:
            result = await agent.execute(task)
            self.failures = 0
            return result
        except Exception as e:
            self.failures += 1
            if self.failures >= self.threshold:
                self.open = True
            raise
```

## Scaling Considerations

### When Scaling Multi-Agent Systems:

1. **Message Queue**: Use Redis/RabbitMQ for async communication
2. **Load Balancing**: Distribute tasks across agent instances
3. **State Store**: Centralize state in database or cache
4. **Monitoring**: Track agent health and performance
5. **Rate Limiting**: Prevent API exhaustion
6. **Logging**: Detailed tracing for debugging

### Architecture for Scale

```
┌─────────────────────────────────────────┐
│          Load Balancer                  │
└────────────┬────────────────────────────┘
             │
    ┌────────┼────────┐
    │        │        │
    ▼        ▼        ▼
┌────────┐┌────────┐┌────────┐
│Agent   ││Agent   ││Agent   │
│Worker  ││Worker  ││Worker  │
│Pool 1  ││Pool 2  ││Pool 3  │
└───┬────┘└───┬────┘└───┬────┘
    │         │         │
    └────┬────┴────┬────┘
         │         │
┌────────▼────────▼────────┐
│   Message Queue (RQ)     │
└────────┬────────┬────────┘
         │        │
┌────────▼──┐┌────▼────────┐
│  State    ││  Monitoring │
│  Store    ││  &  Logging │
│  (Redis)  ││             │
└──────────┘└─────────────┘
```

## Best Practices

1. **Clear Contracts**: Define agent interfaces explicitly
2. **Minimal Coupling**: Agents should be loosely coupled
3. **Timeout Handling**: Always set timeouts on agent calls
4. **Result Validation**: Validate agent outputs
5. **Monitoring**: Track agent metrics and performance
6. **Testing**: Test agents independently and in combination
7. **Documentation**: Clear agent capabilities and limitations
8. **Version Control**: Track agent behavior changes

## Exercise: Build a Multi-Agent Research System

### Task
Create a multi-agent system with:
1. **Router Agent**: Analyzes user query and routes to specialists
2. **Research Agent**: Searches and gathers information
3. **Analyzer Agent**: Processes findings
4. **Synthesizer Agent**: Creates final report

### Requirements
- Implement coordination
- Error handling
- Logging
- Timeout management

## Key Takeaways

- Multi-agent systems require careful architecture design
- Choose architecture based on problem characteristics
- Communication protocols must be well-defined
- Resilience and error handling are critical
- Scaling requires additional infrastructure
- Testing multi-agent systems is more complex
- Monitoring and observability are essential